# 06 Views and Functions

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Use the logic objects of Unity Catalog: <b>views</b> (saved queries, and a way to share a subset of data), <b>materialized views</b> (precomputed, incrementally refreshed results), and <b>functions</b> (SQL scalar UDFs, SQL table functions and Python UDFs registered in the catalog), including how permissions work for each.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Views and catalog functions let you define business logic once and reuse it everywhere, from notebooks, SQL warehouses, dashboards and Genie, with governance. Views are also the classic way to give people access to <b>part</b> of a table. The exam tests the differences between views, temporary views and materialized views.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Logic objects</b><br>
| Object | Stores data? | Refresh | Scope | Typical use |<br>|---|---|---|---|---|<br>| Temporary view | No | Computed on each query | Your session only | Scratch work in a notebook |<br>| View | No | Computed on each query | Catalog object, grantable | Reusable logic, exposing a subset of columns or rows |<br>| Materialized view | Yes (results) | <code>REFRESH</code> or schedule, incremental when possible | Catalog object | Expensive aggregations for dashboards |<br>| SQL function (scalar) | No | n/a | Catalog object (<code>EXECUTE</code>) | Reusable expressions, row filters, masks |<br>| SQL table function | No | n/a | Catalog object | Parameterized views |<br>| Python UDF in UC | No | n/a | Catalog object | Logic that needs Python |<br><br>
<b>Permissions on views</b>: a user needs <code>SELECT</code> on the view (plus <code>USE CATALOG</code> and <code>USE SCHEMA</code>), not on the underlying tables. The view runs with its <b>owner's</b> access to the base tables. That's what makes views useful for sharing a subset of data.
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
Five dashboards each computed "active customer" slightly differently, and their numbers never matched. The team defined <code>gold.is_active_customer(last_order_date DATE)</code> as a catalog SQL function and a view <code>gold.active_customers</code> that uses it. All dashboards and Genie now query the view, the definition lives in one place, and a materialized view precomputes daily active customer counts for the executive dashboard.
</div>

## Syntax

```sql
CREATE OR REPLACE VIEW c.s.v AS SELECT ...;                   -- view
CREATE OR REPLACE TEMP VIEW v AS SELECT ...;                  -- session-scoped
CREATE OR REPLACE MATERIALIZED VIEW c.s.mv AS SELECT ...;     -- stored results
REFRESH MATERIALIZED VIEW c.s.mv;

CREATE OR REPLACE FUNCTION c.s.net(amount DOUBLE, tax DOUBLE)
  RETURNS DOUBLE COMMENT 'Amount without tax' RETURN amount / (1 + tax);

CREATE OR REPLACE FUNCTION c.s.orders_in(country_code STRING)
  RETURNS TABLE (order_id INT, amount DOUBLE)
  RETURN SELECT order_id, amount FROM c.s.orders WHERE country = country_code;

GRANT SELECT ON VIEW c.s.v TO `analysts`;
GRANT EXECUTE ON FUNCTION c.s.net TO `analysts`;
```

## Setup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates an <code>orders</code> table in the <code>governance</code> schema.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
6 orders.
</div>

In [0]:
CATALOG = "workspace"
DB = f"{CATALOG}.governance"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {DB}.vf_orders AS
    SELECT * FROM VALUES
      (1, 'IN', 'asha@example.com', 120.0, DATE'2024-11-01'), (2, 'UK', 'ben@example.com',  80.0, DATE'2024-11-02'),
      (3, 'IN', 'divya@example.com', 45.0, DATE'2024-11-02'), (4, 'AE', 'chen@example.com', 300.0, DATE'2024-11-03'),
      (5, 'UK', 'ben@example.com',   20.0, DATE'2024-11-04'), (6, 'IN', 'asha@example.com',  60.0, DATE'2024-11-05')
    AS t(order_id, country, email, amount, order_date)
""")
print("orders:", spark.table(f"{DB}.vf_orders").count())

## 1. A view for reusable logic and a safe subset

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a view that hides the <code>email</code> column and adds a computed column, then queries it.<br><br>
<b>Why it matters</b><br>The view is a catalog object: anyone with <code>SELECT</code> on it can query it from any compute, without access to <code>vf_orders</code> itself, and they never see <code>email</code>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
6 rows without <code>email</code>, with a <code>size</code> column.
</div>

In [0]:
spark.sql(f"""
    CREATE OR REPLACE VIEW {DB}.vf_orders_public
    COMMENT 'Orders without customer contact details'
    AS SELECT order_id, country, amount, order_date,
              CASE WHEN amount >= 100 THEN 'large' ELSE 'small' END AS size
       FROM {DB}.vf_orders
""")
spark.table(f"{DB}.vf_orders_public").orderBy("order_id").show()

## 2. Views vs temporary views

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a temporary view with the same logic and compares where each one lives.<br><br>
<b>Why it matters</b><br>Temporary views vanish with the session and can't be granted, so they're for scratch work. Anything shared or scheduled should be a view (or a table).
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The view appears in the schema's table list. The temporary view only appears as a temporary object of this session.
</div>

In [0]:
spark.sql(f"CREATE OR REPLACE TEMP VIEW vf_orders_tmp AS SELECT order_id, amount FROM {DB}.vf_orders WHERE amount > 50")
print("in schema:", sorted(t.name for t in spark.catalog.listTables(DB) if t.name.startswith("vf_") and not t.isTemporary))
print("temporary:", [t.name for t in spark.catalog.listTables() if t.isTemporary and t.name.startswith("vf_")])

## 3. A SQL scalar function in the catalog

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a catalog function <code>vf_net_amount(amount, tax_rate)</code> and uses it in a query.<br><br>
<b>Why it matters</b><br>Catalog SQL functions are reusable from any SQL or PySpark code, are optimized like built-ins (they're inlined into the plan), and are governed with <code>EXECUTE</code>.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
Catalog functions work in Free Edition. They're Unity Catalog objects, so this cell runs on Databricks only.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Each order with its net amount at 18% tax.
</div>

In [0]:
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {DB}.vf_net_amount(amount DOUBLE, tax_rate DOUBLE)
    RETURNS DOUBLE
    COMMENT 'Amount excluding tax, rounded to 2 decimals'
    RETURN ROUND(amount / (1 + tax_rate), 2)
""")
display(spark.sql(f"SELECT order_id, amount, {DB}.vf_net_amount(amount, 0.18) AS net FROM {DB}.vf_orders ORDER BY order_id"))

## 4. A SQL table function: a parameterized view

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a function that returns a table of orders for a given country, then queries it like a table.<br><br>
<b>Why it matters</b><br>Table functions behave like views with parameters, which is handy for "orders for country X" without creating one view per value.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
3 orders for <code>IN</code>.
</div>

In [0]:
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {DB}.vf_orders_for(country_code STRING)
    RETURNS TABLE (order_id INT, amount DOUBLE, order_date DATE)
    RETURN SELECT order_id, amount, order_date FROM {DB}.vf_orders WHERE country = country_code
""")
display(spark.sql(f"SELECT * FROM {DB}.vf_orders_for('IN') ORDER BY order_id"))

## 5. A Python UDF registered in Unity Catalog

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Registers a Python function in the catalog and calls it from SQL.<br><br>
<b>Why it matters</b><br>Unlike a notebook-scoped <code>@udf</code>, a catalog Python UDF is shared, versioned in the catalog and governed. Prefer SQL functions when possible (they're faster), and use Python for logic SQL can't express.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
Python UDFs in Unity Catalog need serverless or a recent runtime. If unavailable, the cell prints a message.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Masked emails such as <code>a***@example.com</code>.
</div>

In [0]:
try:
    spark.sql(f"""
        CREATE OR REPLACE FUNCTION {DB}.vf_mask_email(email STRING)
        RETURNS STRING
        LANGUAGE PYTHON
        AS $$
        if email is None or "@" not in email:
            return None
        user, domain = email.split("@", 1)
        return user[0] + "***@" + domain
        $$
    """)
    display(spark.sql(f"SELECT DISTINCT email, {DB}.vf_mask_email(email) AS masked FROM {DB}.vf_orders"))
except Exception as e:
    print("Python UDFs in Unity Catalog not available here:", type(e).__name__, str(e)[:200])

## 6. Materialized views

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a materialized view of revenue per country, queries it, then refreshes it after new data arrives.<br><br>
<b>Why it matters</b><br>A materialized view stores its results, so dashboards read precomputed data instead of re-aggregating the base table on every query. Refreshes are incremental when possible.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
Materialized views are created and refreshed by a serverless pipeline behind the scenes. If creating one from a notebook isn't supported in your workspace, run the same SQL in the SQL editor with a SQL warehouse. The cell prints a message on failure.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Revenue per country, then the <code>IN</code> revenue increases by 500 after the refresh.
</div>

In [0]:
mv = f"{DB}.vf_revenue_mv"
try:
    spark.sql(f"CREATE OR REPLACE MATERIALIZED VIEW {mv} AS SELECT country, SUM(amount) AS revenue FROM {DB}.vf_orders GROUP BY country")
    display(spark.table(mv).orderBy("country"))
    spark.sql(f"INSERT INTO {DB}.vf_orders VALUES (7, 'IN', 'new@example.com', 500.0, DATE'2024-11-06')")
    spark.sql(f"REFRESH MATERIALIZED VIEW {mv}")
    display(spark.table(mv).orderBy("country"))
except Exception as e:
    print("Materialized views not available from this compute:", type(e).__name__, str(e)[:200])

## 7. Inspect and grant

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Lists the functions in the schema, describes one, and shows the grants that would share the view and function.<br><br>
<b>Why it matters</b><br>Functions and views are discovered and governed like tables: <code>SHOW</code>, <code>DESCRIBE</code>, <code>GRANT</code>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The <code>vf_</code> functions, the description of <code>vf_net_amount</code>, and the grant statements.
</div>

In [0]:
display(spark.sql(f"SHOW USER FUNCTIONS IN {DB} LIKE 'vf_*'"))
display(spark.sql(f"DESCRIBE FUNCTION EXTENDED {DB}.vf_net_amount"))

In [0]:
print(f"""
GRANT SELECT  ON VIEW     {DB}.vf_orders_public TO `analysts`;   -- no access to vf_orders needed
GRANT EXECUTE ON FUNCTION {DB}.vf_net_amount   TO `analysts`;
""")

## Under the hood

```
SELECT * FROM vf_orders_public      ─▶ view text expanded into the query ─▶ runs on vf_orders (owner's access)
SELECT vf_net_amount(amount, 0.18)  ─▶ SQL function body inlined into the plan, optimized like a built-in
SELECT * FROM vf_revenue_mv         ─▶ reads stored results (refreshed by a managed pipeline)
```

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Metadata</span> views and functions store definitions, not data. Materialized views also store results.

In [0]:
spark.table(f"{DB}.vf_orders_public").explain()

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: temporary views used in scheduled jobs or shared notebooks</b><br>
They disappear with the session. Use a view or a table.<br><br>
<b>⛔ Problem: granting the base table just so people can use a view</b><br>Not needed in Unity Catalog. Grant <code>SELECT</code> on the view only.<br><br>
<b>⛔ Problem: a Python UDF for simple logic</b><br>A SQL function is inlined and much faster. Keep Python for what SQL can't do.<br><br>
<b>⛔ Problem: dashboards aggregating billions of rows on every refresh</b><br>Use a materialized view (or a Gold table) with precomputed results.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is the difference between a view, a temporary view and a materialized view?</b><br>
A view is a saved query in the catalog, computed every time it's queried, and can be granted. A temporary view is the same but scoped to one session and not stored in the catalog. A materialized view stores its results, which are refreshed on demand or on a schedule (incrementally when possible), so reads are fast.<br><br>

<b>🎤 2. How can you give analysts access to some columns of a table without giving access to the table?</b><br>
Create a view selecting only those columns (or rows) and grant <code>SELECT</code> on the view. In Unity Catalog the view runs with its owner's privileges on the base table, so the analysts don't need access to the table itself.<br><br>

<b>🎤 3. Why define a function in Unity Catalog instead of a notebook UDF?</b><br>
It's defined once and reusable from any notebook, job, SQL warehouse or dashboard, it's governed with <code>EXECUTE</code>, and SQL functions are inlined into the query plan, so they're as fast as built-ins.<br><br>

<b>🎤 4. When would you use a SQL table function?</b><br>
When you need a parameterized view: the same query logic for different input values, such as orders for a given country or date range.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A dashboard runs an expensive aggregation over a large table every time it loads. Which object best reduces the cost while keeping results reasonably fresh?</b><br>
A. A temporary view<br>
B. A materialized view refreshed on a schedule<br>
C. A standard view<br>
D. A Python UDF<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. Which statement about temporary views is true?</b><br>
A. They can be granted to other users<br>
B. They exist only for the current session and are not stored in Unity Catalog<br>
C. They store their results on disk<br>
D. They require <code>CREATE TABLE</code> on the schema<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q3. An analyst has <code>SELECT</code> on a view and <code>USE CATALOG</code> / <code>USE SCHEMA</code> on its parents, but no privileges on the view's base table. Can they query the view?</b><br>
A. No, they need <code>SELECT</code> on the base table<br>
B. Yes, the view accesses the base table with its owner's privileges<br>
C. Only if the view is a materialized view<br>
D. Only from a SQL warehouse<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Create a view <code>vf_large_orders</code> with only orders of 100 or more, and without <code>email</code></li><li>(Databricks) Create a SQL function <code>vf_tax(amount DOUBLE)</code> that returns 18% of the amount, and use it with the view</li><li>(Databricks) Create a table function <code>vf_orders_between(start DATE, end DATE)</code> and query November 2–4</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. A view of large orders without email
spark.sql(f"""
    CREATE OR REPLACE VIEW {DB}.vf_large_orders AS
    SELECT order_id, country, amount, order_date FROM {DB}.vf_orders WHERE amount >= 100
""")
spark.table(f"{DB}.vf_large_orders").orderBy("order_id").show()

In [0]:
# 2. A scalar function used with the view
spark.sql(f"CREATE OR REPLACE FUNCTION {DB}.vf_tax(amount DOUBLE) RETURNS DOUBLE RETURN ROUND(amount * 0.18, 2)")
display(spark.sql(f"SELECT order_id, amount, {DB}.vf_tax(amount) AS tax FROM {DB}.vf_large_orders"))

# 3. A table function with a date range
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {DB}.vf_orders_between(start_date DATE, end_date DATE)
    RETURNS TABLE (order_id INT, amount DOUBLE, order_date DATE)
    RETURN SELECT order_id, amount, order_date FROM {DB}.vf_orders WHERE order_date BETWEEN start_date AND end_date
""")
display(spark.sql(f"SELECT * FROM {DB}.vf_orders_between(DATE'2024-11-02', DATE'2024-11-04') ORDER BY order_id"))

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>View: saved query, grantable, computed on read. Temp view: session only</li><li>Materialized view: stored, refreshable results for expensive queries</li><li>Users need <code>SELECT</code> on a view, not on its base tables</li><li>SQL functions (scalar and table) are inlined and governed with <code>EXECUTE</code>. Python UDFs in UC for logic SQL can't express</li><li>Define business logic once in the catalog, reuse everywhere</li></ul>
</div>

| Object | Create | Grant |
|---|---|---|
| View | `CREATE VIEW c.s.v AS ...` | `GRANT SELECT ON VIEW` |
| Temp view | `CREATE TEMP VIEW v AS ...` | n/a |
| Materialized view | `CREATE MATERIALIZED VIEW c.s.mv AS ...` | `GRANT SELECT` |
| SQL function | `CREATE FUNCTION c.s.f(...) RETURNS t RETURN expr` | `GRANT EXECUTE` |
| Table function | `... RETURNS TABLE (...) RETURN SELECT ...` | `GRANT EXECUTE` |
| Python UDF | `... LANGUAGE PYTHON AS $$ ... $$` | `GRANT EXECUTE` |

## Git commit

```
git add 06_unity_catalog/06_views_and_functions.ipynb
git commit -m "add 06_06 views and functions notebook"
```